# **STAGE 4 - MACHINE LEARNING**

## **Objectives**

* In this notebook, I am going to build and compare machine learning models capable of predicting whether a credit card customer is likely to be attrited.

## **Inputs**

* The input needed for this notebook is the cleaned dataset created in **Stage 2 - ETL**. 
* The relative path for this .csv file is: `datasets/cleaned-data/bank-churners-cleaned.csv`.
* In the notebook, I will outline which variables will be used as features (`X`) for the machine learning models.

## **Outputs**

* The output of this notebook will be two machine learning models fitted and evaluated on their effectiveness in predicting the target variable `Attrition_Flag`.

> *Ethical Considerations*: 
<br><br>*1. How the model is used*
<br><br>First and foremost, the modelling for this section has been undertaken for **educational purposes only** and **should not be used for real-life banking predictions or on real customers**.
<br><br>Whilst predicting churn may not necessarily be problematic, *how* the prediction is used matters. For example, offering a customer a helpful retention benefit is very different from reducing their services, increasing fees, or restricting access because a model predicts they are likely to leave.
<br><br>Customers predicted to churn may include people experiencing financial hardship. Targeting vulnerable customers with aggressive marketing, higher-cost products, or incentives that encourage additional borrowing can cause harm.
<br><br>Ultimately, any churn predictor that is created must be used first and foremost as a means of **improving customer experience** and not to exploit the customer's predicted behaviour and to target them with aggressive marketing.
<br><br>*2. Proxy discrimination*
<br><br>Proxy discrimination is a well-documented issue within banking machine learning. Even if you exclude protected characteristics from a model, there might be correlations that allow a model to essentially develop biases without explicitly being trained with these characteristics as features.
<br><br>

---

## **Change Working Directory**

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'/Users/elliebrawn/Documents/vscode-projects/credit-card-customer-churn-analysis/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'/Users/elliebrawn/Documents/vscode-projects/credit-card-customer-churn-analysis'

---

## **Import Packages and Libraries**

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

# Import Scikit-learn Pipeline
from sklearn.pipeline import Pipeline

# Import ColumnTransformer, which is used to apply different preprocessing steps to different columns of the dataset
from sklearn.compose import ColumnTransformer

# Import OneHotEncoder, which is used to convert categorical variables into binary vectors (one-hot encoding)
from sklearn.preprocessing import OneHotEncoder

# Import StandardScaler, which is used to standardise numerical data to make sure the data points have a balanced scale
from sklearn.preprocessing import StandardScaler

# Import SelectFromModel, which is used for feature selection based on the importance of features determined by a model
from sklearn.feature_selection import SelectFromModel

# Import ML Algorithm Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Import regression metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

---

## **Load Dataset**

Firstly, I will load the cleaned dataset `bank-churners-cleaned.csv`

In [5]:
# Load the cleaned dataset
df_ml = pd.read_csv("datasets/cleaned-data/bank-churners-cleaned.csv")

# Display the first few rows of the dataset to make sure the data has loaded correctly
df_ml.head()

,anonymised_clientnum,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,...,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Attrition_Flag_Binary
0,022e52f0a251431f954db620fd0c87ac3c523c60cc5980...,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,...,3,12691.0,777.0,11914.0,1.335,1144.0,42,1.625,0.061,0
1,2731de4ed9ecb2a3ab828448bfda6137e5c7571e1f7576...,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,...,2,8256.0,864.0,7392.0,1.541,1291.0,33,3.714,0.105,0
2,75dac624c2bbdb15b16abc0350820bcf598c012a76b102...,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,...,0,3418.0,0.0,3418.0,2.594,1887.0,20,2.333,0.000,0
3,1aaada0cd1f7dc23d83a171beec9f398cfac1471841843...,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,...,1,3313.0,2517.0,796.0,1.405,1171.0,20,2.333,0.760,0
4,1811f55b3210153f77e41ceceb61cc181d2edc4e65947e...,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,...,0,4716.0,0.0,4716.0,2.175,816.0,28,2.500,0.000,0


In [6]:
# Display all the columns in my DataFrame to help with feature selection
df_ml.columns

Index(['anonymised_clientnum', 'Attrition_Flag', 'Customer_Age', 'Gender',
       'Dependent_count', 'Education_Level', 'Marital_Status',
       'Income_Category', 'Card_Category', 'Months_on_book',
       'Total_Relationship_Count', 'Months_Inactive_12_mon',
       'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal',
       'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt',
       'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio',
       'Attrition_Flag_Binary'],
      dtype='object')

---

## **1. Machine Learning Algorithm Selection**

The first thing I am going to do is decide what machine learning algorithms are going to be appropriate to predict the target.

| **What are you predicting?** | **Target Variable?** | **Algorithm Type** | **Examples** |
| ----- | ----- | ----- | ----- |
| Continuous Number | Yes | Regression Algorithm | Linear Regression, Decision Tree Regression, Random Forest Regression|
| Category | Yes | Classification Algorithm | Logistic Regression, Decision Tree Classification, Random Forest Classification |
| Category | No | Clustering Algorithm | K-Means Clustering |

Our target `Attrition_Flag_Binary` is a categorical target and we will therefore need a **Classification Algorithm**.

![Algorithm Selection Flow Chart](../images/algorithm-selection-classification.png)

For my prediction model, I am going to be testing the performance of **Logistic Regression** and **Random Forest Classification**.

---

## **2. Split the Dataset into Test and Train**

### **2.1 Define X and y**

Firstly, I am going to define `X` and `y`: in machine learning, `X` represents the input features and `y` represents the target.

For our model, `Attrition_Flag_Binary` is our chosen target variable - `y`.

> *Ethical Considerations*: 
<br><br>*Removing Protected Characteristics from ML*
<br><br>Before I create my ML model, I am going to ensure that I have removed protected characteristics from the features. With reference to the **UK Equality Act 2010**, I have decided to drop `Gender`, `Marital_Status` and `Customer_Age`.
<br><br>*What does this not fully guarantee?*
<br><br>As discussed in previous Jupyter Notebooks, this doesn't guarantee that the model is free of indirect bias via correlated variables. There are definitely still variables that have a plausible proxy to some of the protected characteristics that it is essential that we are aware we have used.
<br><br>

Variables that have not been included as features in `X`:
* `anonymised_clientnum`: Because this is simply a unique identifier (anonymised) for each bank client, it does not offer anything in the way of insight into who is more likely to be an `Existing Customer` vs an `Attrited Customer`.
* `Attrition_Flag`: For obvious reasons, this cannot be included as a feature because it is a direct copy of our target variable, `Attrition_Flag_Binary`.
* `Gender`, `Marital_Status`, `Customer_Age`: These have been removed from the model features list because they are protected characteristics under the **UK Equality Act 2020** and therefore it would not be ethical to include these in the predictor model.

In [7]:
# Define X as the features that will be used to train the model
X = df_ml[[
    "Dependent_count",
    "Education_Level",
    "Income_Category",
    "Card_Category",
    "Months_on_book",
    "Total_Relationship_Count",
    "Months_Inactive_12_mon",
    "Contacts_Count_12_mon",
    "Credit_Limit",
    "Total_Revolving_Bal",
    "Avg_Open_To_Buy",
    "Total_Amt_Chng_Q4_Q1",
    "Total_Trans_Amt",
    "Total_Trans_Ct",
    "Total_Ct_Chng_Q4_Q1",
    "Avg_Utilization_Ratio"
    
]]

# Define y as the target variable that we want to predict
y = df_ml["Attrition_Flag_Binary"]

### **2.2 Split the Dataset**

To split the dataset, I am going to use `test_train_split` from scikit-learn (`sklearn`) to separate the data into a train set (80% of the overall dataset) and a test set (20% of the overall dataset).

In [8]:
# Use test_train_split to split the data into a test set (20%) and a train set (80%)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"The dataset has been split into a Train set and a Test set.\n"
      f"Train set: {X_train.shape}, {y_train.shape}\n"
      f"Test set: {X_test.shape}, {y_test.shape}")

The dataset has been split into a Train set and a Test set.
Train set: (8101, 16), (8101,)
Test set: (2026, 16), (2026,)


---

## **3. Separate Categorical and Numerical Features**

In order to implement a Machine Learning Pipeline, I first need to separate my features into numerical vs categorical features. This allows me to perform the correct feature engineering/ feature scaling steps on the data.

In [9]:
# Select the categorical features from X
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

# Select the numerical features from X
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Ensure that this step has been completed correctly by printing the categorical and numerical features
print(f"The categorical features are: {categorical_features}")
print(f"The numeric features are: {numeric_features}")

The categorical features are: ['Education_Level', 'Income_Category', 'Card_Category']
The numeric features are: ['Dependent_count', 'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal', 'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio']


These features have been split as I would have expected given what we saw in the ETL section and their data types. 

---

## **4. Logistic Regression**

### **4.1 Outline the Machine Learning Pipeline**

> *Notes on Process*: <br><br>Credit is due to the Code Institute LMS for the instructions on creating a pipeline using a method and linking feature engineering, feature scaling, feature selection altogether in one code block.
<br><br>I had originally planned to do all the steps separately but I found it much harder to follow and thus more prone to errors.
<br><br>Additionally to what was taught in the LMS, I used Generative AI (Claude Sonnet 5), to provide me a simplified breakdown of all the steps in a machine learning pipeline and one of the additional tools I was able to learn about was `ColumnTransformer`.
<br><br>According to the [ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) Scikit-learn documentation, it allows different columns or column subsets of the input to be transformed separately - this will be useful because I have some categorical and some numeric column values as my features.
<br><br>

In [10]:
# Create a function to create a Machine Learning Pipeline for Logistic Regression
def pipeline_logistic_regression():
    """
    Create a Machine Learning Pipeline for Logistic Regression.
    
    This function creates a machine learning pipeline, during which the following
    steps are performed:

        1. Feature Engineering
            Preprocessor step established with Column Transformer
            Performs two steps:
            a) Feature Engineering with OneHotEncoder for categorical features
            b) Feature Scaling with StandardScaler for numeric features
        2. Modelling with LogisticRegression
    
    The function returns:
        pipeline: The pipeline object that performs the specified steps.
    """

    # Create a ColumnTransformer to apply OneHotEncoder to categorical features and StandardScaler to numeric features
    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
            ("num", StandardScaler(), numeric_features)
        ]
    )

    # Create a pipeline that includes the preprocessor, feature selection, and the Logistic Regression model
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(class_weight="balanced"))
    ])

    return pipeline

> *Notes on Process*:
<br><br>Because I had already identified that I had a really imbalanced split between my existing customers (`0`) and my attrited customers (`1`), I knew that this would cause issues with my model.
<br><br>Consulting Generative AI (Claude Sonnet 5) led me to `class_weight="balanced"`. What this does is re-weight each class inversely proportional to its frequency in the training data. It essentially pushes the model to pay closer attention to correctly identifying the attrited customers, rather than being able to get away with ignoring them and still ending up with a strong accuracy score on its predictions.
<br><br>

### **4.2 Fit the Pipeline**

Now it's time to fit the pipeline, which allows the model to learn the relationships between the features and the target.

In [11]:
# Fit the Logistic Regression Pipeline to the training data using the estimator .fit()
pipeline_logr = pipeline_logistic_regression()
pipeline_logr.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Education_Level',
                                                   'Income_Category',
                                                   'Card_Category']),
                                                 ('num', StandardScaler(),
                                                  ['Dependent_count',
                                                   'Months_on_book',
                                                   'Total_Relationship_Count',
                                                   'Months_Inactive_12_mon',
                                                   'Contacts_Count_12_mon',
                                                   'Credit_Limit',
                                                   'Total_Revolving_Bal',
                                                   'Avg_Open_To_Buy',
                                                   'Total_Amt_Chng_Q4_Q1',
                                                   'Total_Trans_Amt',
                                                   'Total_Trans_Ct',
                                                   'Total_Ct_Chng_Q4_Q1',
                                                   'Avg_Utilization_Ratio'])])),
                ('model', LogisticRegression(class_weight='balanced'))])

### **4.3 Evaluate the Model**

Because I have used `class_weight="balanced"`, I expect that:
* The model has become more willing to predict `1` (attrited) even when it is less certain. Typically, this would mean that it catches more true attriters (**increases recall**) but tends to get more false positives (**decreases precision**).
* This trade-off is one that seems acceptable - it would be better to mistakenly flag a loyal customer for more attention and customer service than to miss a genuine at-risk customer.

The metrics I intend to use to measure how well the model has worked in predicting attrited customers are:

| **Model Evaluation Metric** | **Explanation** | **Scale** | **Best Direction** |
| --------------------------- | ----------------| ---------------- | ------------------ |
| **Accuracy** | This tells us the proportion of the predictions that were correct (in this instance, for Attrited - `1` and Existing - `0`)| 0 to 1 | The higher the better, although as we have identified already, this metric is not useful on its own particularly with imbalances. For example, with the 84% / 16% imbalance, if the model simply predicted Existing - `0` for everything, it would still get an accuracy score of 0.84 (84%). |
| **Precision** | This tells us for every `1` that the model predicted, what proportion of these actually *were* `1`s. | 0 to 1 | Higher is better - the higher the prediction metric, the fewer false alarms (in this case, a false alarm would mean existing customers are wrongly flagged as at-risk). |
| **Recall** | Recall is the percentage of the class that was properly predicted - in other words, of the customers who *actually* attrited, what proportion did the model correctly identify. | 0 to 1 | A higher recall means fewer missed false negatives (in this case, our attrited customers). In the case of identifying our bank churners, recall should be looked at as the priority over predicions, because as we identified earlier, it would be worse for us to miss a genuine at-risk customer. |
| **F1 Score** | The F1 score averages out the recall and precision scores. It's useful when you care about both false positives and false negatives. | 0 to 1 | Higher is better. |

In [12]:
# Step 1: Make Predictions

prediction_train_logr = pipeline_logr.predict(X_train)

prediction_test_logr = pipeline_logr.predict(X_test)

# Step 2: Evaluate the Test Set

test_logr_accuracy = accuracy_score(y_test, prediction_test_logr)
test_logr_recall = recall_score(y_test, prediction_test_logr)
test_logr_precision = precision_score(y_test, prediction_test_logr)
test_logr_f1 = f1_score(y_test, prediction_test_logr)

print("="*45)
print("LOGISTIC REGRESSION - TEST SET EVALUATION")
print("="*45)
print(f"Accuracy: {test_logr_accuracy.round(3)}")
print(f"Recall: {test_logr_recall.round(3)}")
print(f"Precision: {test_logr_precision.round(3)}")
print(f"F1 Score: {test_logr_f1.round(3)}")
print("")

# Step 3: Evaluate the Train Set (Comparison Check to see if the model is overfitting)
train_logr_accuracy = accuracy_score(y_train, prediction_train_logr)
train_logr_recall = recall_score(y_train, prediction_train_logr)
train_logr_precision = precision_score(y_train, prediction_train_logr)
train_logr_f1 = f1_score(y_train, prediction_train_logr)

print("="*45)
print("LOGISTIC REGRESSION - TRAIN SET EVALUATION")
print("="*45)
print(f"Accuracy: {train_logr_accuracy.round(3)}")
print(f"Recall: {train_logr_recall.round(3)}")
print(f"Precision: {train_logr_precision.round(3)}")
print(f"F1 Score: {train_logr_f1.round(3)}")
print("")

LOGISTIC REGRESSION - TEST SET EVALUATION
Accuracy: 0.851
Recall: 0.844
Precision: 0.525
F1 Score: 0.647

LOGISTIC REGRESSION - TRAIN SET EVALUATION
Accuracy: 0.843
Recall: 0.85
Precision: 0.506
F1 Score: 0.635



### **4.4 Summary of Logistic Regression Model Performance**

#### **4.4.1 Train vs Test Set Comparison**

The scores for train and test set are fairly closely matched, meaning that there is no indication of the model overfitting.

#### **4.4.2 Interpreting the Scores (Test Set)**

| **Metric** | **Score** | **What does this mean?** |
| ---------- | --------- | ------------------------ |
| Accuracy | 0.851 | 85.1% of all predictions for both Existing and Attrited customers were correct. However, as we've already identified, this score alone will not be very useful in identifying the performance of the model because of the imbalance in the two classes. |
| Recall | 0.844 | Of all the customers who *actually* attrited, the model identified 84.4% of them. This is strong and appears to show that the `class_weight="balanced"` worked as I hoped it would - the model is able to catch a large proportion of the attrited customers |
| Precision | 0.525 | As I noted when I decided to use `class_weight="balanced"`, the precision score was likely to be the score most affected. We prioritised recall and so a large amount of false positives have been generated by the model - of the customers that the model flagged as being attrited - `1`, only 52.5% actually were. |
| F1 | 0.647 | This is a moderate score and is understandably so because it reflects the trade-off of a high recall score and a lower precision score. |

---

## **5. Random Forest Classification**

### **5.1 Outline the Machine Learning Pipeline**

In [13]:
# Create a function to create a Machine Learning Pipeline for Random Forest Classification
def pipeline_random_forest_classification():
    """
    Create a Machine Learning Pipeline for Random Forest Classification.
    
    This function creates a machine learning pipeline, during which the following
    steps are performed:

        1. Feature Engineering
            Preprocessor step established with Column Transformer
            Performs two steps:
            a) Feature Engineering with OneHotEncoder for categorical features
            b) Feature Scaling with StandardScaler for numeric features
        2. Modelling with RandomForestClassifier 
    
    The function returns:
        pipeline: The pipeline object that performs the specified steps.
    """

    # Create a ColumnTransformer to apply OneHotEncoder to categorical features and StandardScaler to numeric features
    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
            ("num", StandardScaler(), numeric_features)
        ]
    )

    # Create a pipeline that includes the preprocessor, feature selection, and the Logistic Regression model
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            random_state=42,
            class_weight="balanced",
            max_depth=10,
            min_samples_leaf=5,
            min_samples_split=10
            ))
    ])

    return pipeline

> *Troubleshooting Issues*:
<br><br>The first time I set up this Random Forest Classification pipeline, fitted it and then evaluated the model, I had only included
<br>`("model", RandomForestClassifier(random_state=42, class_weight="balanced"))`
<br><br>This led the unconstrained tree to completely memorise the training data and therefore perform less effectively on the test data - I got perfect `1.0`s against every metric in my evaluation of the train set. I didn't want to leave this as is and simply conclude that Logistic Regression was the best model without first identifying if there were things I could do in the pipeline which would help me to prevent this overfitting from happening. 
<br><br>Generative AI (Claude Sonnet 5) was a real help in identifying different hyperparameters to help me adjust my model.
<br><br>

| **Hyperparameter** | **What does it do?** |
| ------------------ | ---------------------|
| `max_depth` | Caps how many levels each tree can grow - this means that each tree has to generalise and isn't simply free to create more and more specific rules for smaller and smaller subsets of the data. |
| `min_samples_leaf` | This says that a leaf (an end node) must contain a specified number of data points and can't simply contain one data point. |
| `min_samples_split` | This says that a node has to have a specified number of samples before it can split further. |

### **5.2 Fit the Pipeline**

In [14]:
# Fit the Random Forest Classification Pipeline to the training data using the estimator .fit()
pipeline_rf = pipeline_random_forest_classification()
pipeline_rf.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Education_Level',
                                                   'Income_Category',
                                                   'Card_Category']),
                                                 ('num', StandardScaler(),
                                                  ['Dependent_count',
                                                   'Months_on_book',
                                                   'Total_Relationship_Count',
                                                   'Months_Inactive_12_mon',
                                                   'Contacts_Count_12_mon',
                                                   'Credit_Limit',
                                                   'Total_Revolving_Bal',
                                                   'Avg_Open_To_Buy',
                                                   'Total_Amt_Chng_Q4_Q1',
                                                   'Total_Trans_Amt',
                                                   'Total_Trans_Ct',
                                                   'Total_Ct_Chng_Q4_Q1',
                                                   'Avg_Utilization_Ratio'])])),
                ('model',
                 RandomForestClassifier(class_weight='balanced', max_depth=10,
                                        min_samples_leaf=5,
                                        min_samples_split=10,
                                        random_state=42))])

### **5.3 Evaluate the Model**

In [15]:
# Step 1: Make Predictions

prediction_train_rf = pipeline_rf.predict(X_train)

prediction_test_rf = pipeline_rf.predict(X_test)

# Step 2: Evaluate the Test Set

test_rf_accuracy = accuracy_score(y_test, prediction_test_rf)
test_rf_recall = recall_score(y_test, prediction_test_rf)
test_rf_precision = precision_score(y_test, prediction_test_rf)
test_rf_f1 = f1_score(y_test, prediction_test_rf)

print("="*53)
print("RANDOM FOREST CLASSIFICATION - TEST SET EVALUATION")
print("="*53)
print(f"Accuracy: {test_rf_accuracy.round(3)}")
print(f"Recall: {test_rf_recall.round(3)}")
print(f"Precision: {test_rf_precision.round(3)}")
print(f"F1 Score: {test_rf_f1.round(3)}")
print("")

# Step 3: Evaluate the Train Set (Comparison Check to see if the model is overfitting)
train_rf_accuracy = accuracy_score(y_train, prediction_train_rf)
train_rf_recall = recall_score(y_train, prediction_train_rf)
train_rf_precision = precision_score(y_train, prediction_train_rf)
train_rf_f1 = f1_score(y_train, prediction_train_rf)

print("="*53)
print("RANDOM FOREST CLASSIFICATION - TRAIN SET EVALUATION")
print("="*53)
print(f"Accuracy: {train_rf_accuracy.round(3)}")
print(f"Recall: {train_rf_recall.round(3)}")
print(f"Precision: {train_rf_precision.round(3)}")
print(f"F1 Score: {train_rf_f1.round(3)}")
print("")

RANDOM FOREST CLASSIFICATION - TEST SET EVALUATION
Accuracy: 0.948
Recall: 0.872
Precision: 0.817
F1 Score: 0.843

RANDOM FOREST CLASSIFICATION - TRAIN SET EVALUATION
Accuracy: 0.966
Recall: 0.975
Precision: 0.838
F1 Score: 0.902



### **5.4 Summary of Random Forest Classification Model Performance**

#### **5.4.1 Train vs Test Set Comparison**

I identified overfitting in the first test of the Random Forest Classifier because the model was unconstrained. By applying hyperparameters to my unconstrained model, the new results for the test and train set are much more positive. 

Whilst there still is a slight discrepancy between the two sets of results, this is much more normal and is to be expected - we'd ultimately expect that the model would perform better on a train set than on the unseen test set data.

#### **5.4.2 Interpreting the Scores (Test Set)**

| **Metric** | **Score** | **What does this mean?** |
| ---------- | --------- | ------------------------ |
| Accuracy | 0.948 | 94.8% of all predictions for both Existing and Attrited customers on the unseen data were correct (although as we mentioned in the last section, the imbalance between the two classes makes accuracy not that good an indicator of the model performance. |
| Recall | 0.872 | The model correctly identified 87.2% of the customers who *actually* attrited. This is another strong result and one that has improved on Logistic Regression's 0.844. |
| Precision | 0.817 | This result is substantially higher than the recall result we got for Logistic Regression. Of the customers that the model predicted would be attrited, 81.7% of them actually were. |
| F1 | 0.843 | This score reflects a strong balance between precision and recall. |

---

## **6. Conclusion**

When evaluating the performance of the two models side-by-side, we can see that the more complex tree-based algorithm outperformed the Logistic Regression model on every metric. 

| **Performance (Test Set)** | **Logistic Regression** | **Random Forest** |
| -------------------------- | ----------------------- | ----------------- |
| Accuracy | 0.851 | 0.948 |
| Recall | 0.844 | 0.872 |
| Precision | 0.525 | 0.817 |
| F1 | 0.647 | 0.843 |


**The Random Forest model offers a stronger and also more balanced prediction model** that not only correctly identifies more attrited customers, but also avoids as many false-alarms. From a business perspective, the higher precision score means the bank would be able to spend retention resources more efficiently and contact far fewer non-attriting customers unnecessarily, while the high recall score means it would be likely to miss less actual attriters.

---